In [8]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
from pathlib import Path

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions, RunningMode

In [9]:
# Paths
RAW_VIDEO_DIR = Path("../data/raw/video_data")
MODEL_PATH    = Path("../data/models/pose_landmarker_lite.task")
OUTPUT_CSV    = Path("../data/raw/tabular_data/angles_dataset.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Exercise folder names
EXERCISES = ["pullup", "pushup", "russian_twist", "squat"]

# Subfolders per exercise
CORRECTNESS_LABELS = ["correct", "incorrect"]

print("✅ Setup complete.")
print(f"   Model        : {MODEL_PATH.resolve()}")
print(f"   Video source : {RAW_VIDEO_DIR.resolve()}")
print(f"   Output CSV   : {OUTPUT_CSV.resolve()}")

✅ Setup complete.
   Model        : C:\Users\USER\Desktop\AI_PROJECT_2\data\models\pose_landmarker_lite.task
   Video source : C:\Users\USER\Desktop\AI_PROJECT_2\data\raw\video_data
   Output CSV   : C:\Users\USER\Desktop\AI_PROJECT_2\data\raw\tabular_data\angles_dataset.csv


In [10]:
import sys
sys.path.insert(0, "..")
from src.shared_utils import calculate_angle, LANDMARKS, get_coords, get_z, extract_angles

# Sanity check
test_angle = calculate_angle((0, 0), (1, 0), (1, 1))
print(f"Shared utilities loaded. Sanity check (expect 90): {test_angle}°")


Shared utilities loaded. Sanity check (expect 90): 90.0°


In [11]:
# LANDMARKS, get_coords, get_z, extract_angles are imported from src/shared_utils.py above.
print(f"   Joints tracked : {list(LANDMARKS.keys())}")
print(f"   Angles output  : 10 joint angles + 3 torso rotation features = 13 features per frame")


   Joints tracked : ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle', 'left_heel', 'right_heel']
   Angles output  : 10 joint angles + 3 torso rotation features = 13 features per frame


In [12]:
def process_video(video_path, exercise_name, exercise_correctness, options, video_id):
    """
    Process a single video file and return a list of rows,
    where each row represents one frame's angles + metadata.

    Args:
        video_path          : Path to the video file.
        exercise_name       : e.g. "squat"
        exercise_correctness: "correct" or "incorrect"
        options             : PoseLandmarkerOptions (initialized outside)

    Returns:
        List of dicts, one per valid frame.
    """
    rows = []
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"   ⚠️  Could not open: {video_path.name}")
        return rows

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        import warnings
        warnings.warn(f"[{video_path.name}] Could not read FPS from video metadata; defaulting to 30 fps. Velocity values may be inaccurate.", RuntimeWarning)
        fps = 30  # fallback — matches the 30 fps of all training videos

    prev_angles = None
    frame_number = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Convert frame to MediaPipe Image
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            # Timestamp in milliseconds (required for VIDEO mode)
            timestamp_ms = int((frame_number / fps) * 1000)

            # Run pose detection
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]  # first (and usually only) person
                angles    = extract_angles(landmarks)

                # --- Angular velocities (change in angle per second) ---
                if prev_angles is not None:
                    angular_velocities = {
                        f"{key}_velocity": round(abs(angles[key] - prev_angles[key]) * fps, 2)
                        for key in angles
                    }
                else:
                    # First valid frame: velocity is 0
                    angular_velocities = {f"{key}_velocity": 0.0 for key in angles}

                row = {
                    "video_id"            : video_id,
                    "frame_number"        : frame_number,
                    **angles,
                    **angular_velocities,
                    "exercise_name"       : exercise_name,
                    "exercise_correctness": exercise_correctness,
                }
                rows.append(row)
                prev_angles = angles

            frame_number += 1

    cap.release()
    return rows


print("✅ Video processing function ready.")
print("   Outputs per frame: frame_number + 10 angles + 10 angular velocities + 2 labels")

✅ Video processing function ready.
   Outputs per frame: frame_number + 10 angles + 10 angular velocities + 2 labels


In [13]:
def build_dataset(raw_video_dir, exercises, correctness_labels):
    """
    Walk through all exercise/correctness subfolders,
    process every video, and return the full combined DataFrame.
    """
    all_rows = []
    video_id_counter = 0

    # Initialize PoseLandmarker options once (shared across all videos)
    base_options = mp_python.BaseOptions(model_asset_path=str(MODEL_PATH))
    options = PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=RunningMode.VIDEO,
        num_poses=1,                  # we only expect one person per video
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )

    for exercise in exercises:
        for correctness in correctness_labels:
            folder = raw_video_dir / exercise / correctness

            if not folder.exists():
                print(f"⚠️  Folder not found, skipping: {folder}")
                continue

            video_files = list(folder.glob("*.mp4")) + list(folder.glob("*.avi")) + list(folder.glob("*.mov"))

            if not video_files:
                print(f"⚠️  No videos found in: {folder}")
                continue

            print(f"\n📂 {exercise} / {correctness}  ({len(video_files)} videos)")

            for video_path in video_files:
                print(f"   🎞️  Processing: {video_path.name} ...", end=" ")
                rows = process_video(video_path, exercise, correctness, options, video_id_counter)
                all_rows.extend(rows)
                print(f"{len(rows)} frames extracted.")
                video_id_counter += 1

    df = pd.DataFrame(all_rows)
    return df


# --- Run it ---
print("🚀 Starting dataset creation...\n")
df_raw = build_dataset(RAW_VIDEO_DIR, EXERCISES, CORRECTNESS_LABELS)

print(f"\n✅ Done! Total frames collected: {len(df_raw)}")
print(f"   Shape : {df_raw.shape}")
print(f"   Columns: {list(df_raw.columns)}")

🚀 Starting dataset creation...


📂 pullup / correct  (39 videos)
   🎞️  Processing: Fragment 1.mp4 ... 64 frames extracted.
   🎞️  Processing: Fragment 10.mp4 ... 78 frames extracted.
   🎞️  Processing: Fragment 11.mp4 ... 72 frames extracted.
   🎞️  Processing: Fragment 12.mp4 ... 77 frames extracted.
   🎞️  Processing: Fragment 13.mp4 ... 65 frames extracted.
   🎞️  Processing: Fragment 15.mp4 ... 57 frames extracted.
   🎞️  Processing: Fragment 16.mp4 ... 72 frames extracted.
   🎞️  Processing: Fragment 17.mp4 ... 63 frames extracted.
   🎞️  Processing: Fragment 18.mp4 ... 61 frames extracted.
   🎞️  Processing: Fragment 19.mp4 ... 63 frames extracted.
   🎞️  Processing: Fragment 2.mp4 ... 74 frames extracted.
   🎞️  Processing: Fragment 20.mp4 ... 60 frames extracted.
   🎞️  Processing: Fragment 21.mp4 ... 66 frames extracted.
   🎞️  Processing: Fragment 22.mp4 ... 68 frames extracted.
   🎞️  Processing: Fragment 23.mp4 ... 67 frames extracted.
   🎞️  Processing: Fragment 24.mp4 ..

In [14]:
# --- Save to CSV ---
df_raw.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Raw dataset saved to: {OUTPUT_CSV}")
print(f"   Shape: {df_raw.shape}\n")

# --- Sanity Checks ---

# 1. Preview
print("=" * 60)
print("📋 First 5 rows:")
display(df_raw.head())

# 2. Class distribution
print("=" * 60)
print("📊 Exercise distribution:")
display(df_raw["exercise_name"].value_counts())

print("\n📊 Correctness distribution:")
display(df_raw["exercise_correctness"].value_counts())

print("\n📊 Exercise × Correctness breakdown:")
display(df_raw.groupby(["exercise_name", "exercise_correctness"]).size().unstack(fill_value=0))

# 3. Null check
print("=" * 60)
null_counts = df_raw.isnull().sum()
if null_counts.sum() == 0:
    print("✅ No null values found.")
else:
    print("⚠️  Null values detected:")
    display(null_counts[null_counts > 0])

# 4. Angle range check (all angles should be between 0 and 180)
print("=" * 60)
angle_cols = [col for col in df_raw.columns if "angle" in col and "velocity" not in col]
out_of_range = df_raw[angle_cols].apply(lambda col: ((col < 0) | (col > 180)).sum())

if out_of_range.sum() == 0:
    print("✅ All angles are within [0°, 180°].")
else:
    print("⚠️  Out-of-range angles detected:")
    display(out_of_range[out_of_range > 0])

# 5. Basic statistics
print("=" * 60)
print("📈 Angle columns statistics:")
display(df_raw[angle_cols].describe())

✅ Raw dataset saved to: ..\data\raw\tabular_data\angles_dataset.csv
   Shape: (27456, 30)

📋 First 5 rows:


,video_id,frame_number,left_elbow_angle,right_elbow_angle,left_shoulder_angle,right_shoulder_angle,left_hip_angle,right_hip_angle,left_knee_angle,right_knee_angle,...,right_hip_angle_velocity,left_knee_angle_velocity,right_knee_angle_velocity,left_ankle_angle_velocity,right_ankle_angle_velocity,shoulder_z_diff_velocity,hip_z_diff_velocity,torso_rotation_velocity,exercise_name,exercise_correctness
0,0,0,165.92,176.17,160.69,173.34,168.20,173.08,176.95,171.07,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,pullup,correct
1,0,1,167.38,178.32,160.27,171.76,168.42,173.78,177.05,175.20,...,21.0,3.0,123.9,87.6,376.2,0.29,0.12,0.16,pullup,correct
2,0,2,166.59,179.51,158.92,169.74,168.47,174.47,177.16,176.46,...,20.7,3.3,37.8,234.6,13.2,2.01,1.17,0.84,pullup,correct
3,0,3,165.42,177.20,158.65,168.31,167.29,174.86,176.81,177.64,...,11.7,10.5,35.4,127.8,60.9,2.54,0.03,2.51,pullup,correct
4,0,4,166.39,173.75,158.61,165.83,167.67,174.92,176.35,179.51,...,1.8,13.8,56.1,483.9,106.5,0.39,0.26,0.13,pullup,correct


📊 Exercise distribution:


exercise_name
russian_twist    7275
pushup           7039
squat            6834
pullup           6308
Name: count, dtype: int64


📊 Correctness distribution:


exercise_correctness
incorrect    13825
correct      13631
Name: count, dtype: int64


📊 Exercise × Correctness breakdown:


exercise_correctness,correct,incorrect
exercise_name,,
pullup,3007,3301
pushup,3628,3411
russian_twist,3643,3632
squat,3353,3481


✅ No null values found.
✅ All angles are within [0°, 180°].
📈 Angle columns statistics:


,left_elbow_angle,right_elbow_angle,left_shoulder_angle,right_shoulder_angle,left_hip_angle,right_hip_angle,left_knee_angle,right_knee_angle,left_ankle_angle,right_ankle_angle
count,27456.000000,27456.000000,27456.000000,27456.000000,27456.000000,27456.000000,27456.000000,27456.000000,27456.000000,27456.000000
mean,119.361214,117.251677,66.316198,68.040743,127.587192,129.427377,138.636608,139.006342,144.180805,146.985035
std,44.541901,46.304342,49.167699,47.408438,51.408627,50.663986,41.792427,43.374345,16.799077,15.236524
min,0.020000,0.010000,0.000000,0.000000,6.360000,2.240000,8.790000,0.150000,4.350000,8.910000
25%,88.950000,81.515000,18.100000,24.060000,61.462500,63.587500,92.847500,91.717500,131.780000,135.547500
50%,130.110000,128.050000,62.875000,62.000000,154.690000,155.020000,160.100000,161.760000,143.645000,145.690000
75%,157.020000,156.830000,97.350000,102.482500,168.040000,168.082500,172.090000,172.960000,156.032500,156.692500
max,180.000000,180.000000,179.990000,179.920000,180.000000,180.000000,180.000000,180.000000,179.990000,180.000000
